### Ollama DocChat GenAI App Using LangChain

In [1]:
# Langchain imports
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Community integrations
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# Ollama integrations
from langchain_ollama import ChatOllama

# Environment Variables
from dotenv import load_dotenv

load_dotenv()

c:\Users\viren\anaconda3\envs\langchain_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\viren\AppData\Local\Temp\ipykernel_17736\3214160111.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [2]:
# Scraping the target website for the document
loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewEngineTraceDebugObserveQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdvanced tracing techniquesData & privacyTroubleshooting guidesReferenceOverviewLangSmith Python SDKLangSmith JS/TS SDKLangSmith Go SDKLa

In [3]:
# Splitting the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'description': 'Add LangSmith observability to an LLM application across prototyping, beta testing, and production.', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialOverviewEngineTraceDebugObserveQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdvanced tracing techniquesData & privacyTroubleshooting guidesReferenceOverviewLangSmith Python SDKLangSmith JS/TS SDKLangSmith Go SDKLa

In [4]:
# Converting the chunks into vectors and storing it in the vector store
embeddings = OllamaEmbeddings(model='qwen3-embedding:4b')
vector_store = FAISS.from_documents(documents, embeddings)

C:\Users\viren\AppData\Local\Temp\ipykernel_17736\2955292831.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model='qwen3-embedding:4b')


In [5]:
# Storing the vector store locally and loading from it
vector_store.save_local("faiss_index")
vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

In [6]:
# Converting the vector store into retriever
retriever = vector_store.as_retriever()

In [7]:
# Initializing the ollama llm model
model = ChatOllama(model="qwen3:8b")

In [8]:
# Creating the custom prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            """
            You are an expert AI assistant. 
            Answer the question based ONLY on the provided context. Context: {context}
            """
        ),
        ("user", "{input}")
    ]
)

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n            You are an expert AI assistant. \n            Answer the question based ONLY on the provided context. Context: {context}\n            '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [9]:
# Initializing the output parser 
output_parser = StrOutputParser()

In [10]:
# Creating the lcel chain
chain = ( {"context": retriever, "input": RunnablePassthrough()} | prompt | model | output_parser)

In [11]:
# Querying with the input query by invoking the chain
input_query = "Summarize the document"
response = chain.invoke(input_query)
clean_response = response.replace("**", "").replace("\\n", "\n").strip()
print(clean_response)

The document provides a tutorial on integrating LangSmith observability into an LLM application (e.g., a customer support chatbot) to monitor and trace its behavior across development, testing, and production. Key steps include:

1. Setup:  
   - Wrap an OpenAI client with `wrap_openai` for tracing.  
   - Define a retriever function (`@traceable`) to fetch documents (e.g., support policies) based on user queries.  

2. Support Bot Logic:  
   - A `support_bot` function uses the retriever to generate responses by combining retrieved documents with a system message.  
   - Example: Answers questions like *"How many users can I have on the Starter plan?"* using predefined documents.  

3. Tracing:  
   - Decorate functions (e.g., `@traceable`) to log LLM calls and pipeline steps (retrieval + generation).  
   - Traces are visible via the LangSmith UI or CLI (`langsmith trace list`).  

4. Use Cases:  
   - Prototyping: Track RAG (Retrieval-Augmented Generation) pipelines.  
   - Beta Tes